# 06 — Target B: Teacher-Facing Risk Level

**Question:** given what we know about a learner up to intervention session `t`, how urgently does
this learner need teacher attention **right now, relative to their classmates at the same session**?

This is the product this component exists to produce. Kernel `Python (c3-ews)`, seed `20260828`.
Reads `outputs/models/nb05_primary_logreg_armA.joblib` (the **uncalibrated** primary model — NB05b
found calibration's Brier gain on the 2+ class was not distinguishable from noise once bootstrapped
by learner, 95% CI `[-0.0033, +0.0004]`, while costing 0.043 macro-F1) and
`outputs/tables/24_engineered_feature_matrix.csv`.

## Owner decisions this notebook implements, not revisits

1. **Rank-based, not probability cuts** (NB05b §1–2). The calibration pass did not survive a
   by-learner bootstrap; the uncalibrated model ranks as well and classifies better.
2. **Uncalibrated model.** `nb05_primary_logreg_armA.joblib`.
3. **Risk levels are current-snapshot triage**, not persistent classifications. Every label,
   docstring and output column reflects that framing — see §F for why (rank-decay).
4. **No binary "needs attention" classifier** (owner decision 5-Q2). Flagging everyone scores
   F2 0.877 against the model's 0.689 on that question, so a flag/no-flag claim is not defensible.

No cluster features, no PCA feature space, no `population`/`dd_subtype` as predictors — `population`
is used for evaluation stratification only, from the quarantined audit labels.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap

from nbconfig import *

np.random.seed(RANDOM_SEED)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)
print("seed:", RANDOM_SEED)
print("tables dir:", TABLES_DIR)


seed: 20260828
tables dir: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\tables


## A. Load the primary model, reproduce NB05/NB05b's split, score the TEST set

Same load as NB05 §A / NB05b §A: `24_engineered_feature_matrix.csv`, target collapsed 0/1/2+,
learner-grouped `train`/`test` split materialised in `split`, `population` merged in from the
quarantined audit labels for evaluation only. Scored with `nb05_primary_logreg_armA.joblib`
(uncalibrated), and cross-checked against NB05's saved `57_test_predicted_probabilities.csv`.

In [2]:
df = pd.read_csv(TABLES_DIR / "24_engineered_feature_matrix.csv")
audit = pd.read_csv(AUDIT_LABELS_PATH)[["student_id", "population"]]

df["target"] = df["future_stagnant_count"].clip(upper=2).astype(int)
CLASS_NAMES = {0: "0", 1: "1", 2: "2+"}
CLASSES = [0, 1, 2]

df = df.merge(audit, on="student_id", how="left")
assert df["population"].isna().sum() == 0

META_COLS = ["student_id", "t", "split", "future_stagnant_count", "target", "population"]
FEATURES_A = [c for c in df.columns if c not in META_COLS]

train, test = df[df.split == "train"].reset_index(drop=True), df[df.split == "test"].reset_index(drop=True)
assert train.student_id.nunique() == 210 and test.student_id.nunique() == 90
assert set(train.student_id) & set(test.student_id) == set(), "student_id leaked across the split"

armA_pipe = joblib.load(MODELS_DIR / "nb05_primary_logreg_armA.joblib")
proba = armA_pipe.predict_proba(test[FEATURES_A])
pred = armA_pipe.predict(test[FEATURES_A])

scored = test[["student_id", "t", "population", "target"]].copy()
scored["score_2plus"] = proba[:, 2]
scored["score_1"] = proba[:, 1]
scored["score_0"] = proba[:, 0]
scored["actual_class"] = scored["target"].map(CLASS_NAMES)

# cross-check against NB05's saved artefact
saved = pd.read_csv(TABLES_DIR / "57_test_predicted_probabilities.csv")
check = scored.merge(saved, on=["student_id", "t"], suffixes=("", "_saved"))
assert np.allclose(check["score_2plus"], check["prob_class_2+"], atol=1e-9)
print(f"scored {len(scored)} test rows across {scored.student_id.nunique()} learners — matches NB05's saved probabilities.")
scored.head()


scored 810 test rows across 90 learners — matches NB05's saved probabilities.


,student_id,t,population,target,score_2plus,score_1,score_0,actual_class
0,STU-0001,3,TD,1,0.000799,0.428692,0.570509,1
1,STU-0001,5,TD,1,0.177442,0.529973,0.292585,1
2,STU-0001,7,TD,1,0.403410,0.379529,0.217061,1
3,STU-0001,9,TD,1,0.349128,0.481176,0.169697,1
4,STU-0001,11,TD,1,0.273559,0.393360,0.333081,1


## B. Rank WITHIN each snapshot `t` — the central implementation requirement

`t` is the strongest single feature in the set (NB02: MI 0.0743) and 2+ prevalence rises from 0.000
at `t=3` to 0.144 at `t=19`. A ranking pooled across `t` would fill the High tier almost entirely with
late-`t` learners and leave early sessions empty — reintroducing the calendar-drift problem NB02, NB03
and NB05 all worked to control. Ranking **within** each `t` is what makes "relative priority at the
current session" true rather than aspirational.

`risk_pct` is each learner's percentile rank of `score_2plus` **among the other TEST learners at the
same `t`** (1.0 = highest risk at that snapshot).

In [3]:
scored["risk_pct"] = scored.groupby("t")["score_2plus"].rank(pct=True, method="average")

# every t must have the same number of test learners for a percentile cut to mean the same
# thing at every snapshot
per_t_n = scored.groupby("t").size()
print("test learners per t:")
print(per_t_n)
assert per_t_n.nunique() == 1, "within-t ranking requires equal n per t for a top-k% cut to be comparable across t"
print(f"\nConfirmed: every t has exactly {per_t_n.iloc[0]} test learners, so a top-k% cut selects the same COUNT at every t.")


test learners per t:
t
3     90
5     90
7     90
9     90
11    90
13    90
15    90
17    90
19    90
dtype: int64

Confirmed: every t has exactly 90 test learners, so a top-k% cut selects the same COUNT at every t.


## C. The cut candidates

**Why not the raw count.** Doc 08 §6 expects High at roughly 8–15% of learners. No cut on the raw
predicted count lands there: `count≥2` gives 6.0%, `count≥1` gives 55.1% — this was escalated as an
open blocker in NB02. A top-k% rank rule hits any target rate **by construction**, which is a real
benefit of the rank-based design and resolves that blocker outright.

In [4]:
raw_ge2 = (df["target"] == 2).mean()
raw_ge1 = (df["target"] >= 1).mean()
print(f"Raw count>=2 rate (whole engineered matrix): {raw_ge2:.1%}")
print(f"Raw count>=1 rate (whole engineered matrix): {raw_ge1:.1%}")
print("Neither lands in the doc 08 8-15% band for High -- this is the NB02 blocker a top-k% rule resolves by construction.")


Raw count>=2 rate (whole engineered matrix): 6.0%
Raw count>=1 rate (whole engineered matrix): 55.1%
Neither lands in the doc 08 8-15% band for High -- this is the NB02 blocker a top-k% rule resolves by construction.


### C.1 Operating window — the two tiers do not share a start `t`

From NB05/NB05b: a "≥1 watch" signal is usable from `t=3` (lift 2.70), but the 2+ "High" tier is not
definable before `t~9-11` — 2+ prevalence is 0.000 at `t=3`, 0.011 at `t=5`, 0.022 at `t=7`. **No
High tier is emitted before `t=9`**; a High label at `t=5` would be a claim the data cannot support.
Before `t=9`, only Low/Moderate are distinguished, on the same underlying rank.

In [5]:
HIGH_ELIGIBLE_MIN_T = 9

def assign_tiers(scored_df, high_frac, moderate_frac):
    # top-`high_frac` -> High, next `moderate_frac` -> Moderate, remainder -> Low, all within-t.
    # High is suppressed (downgraded to Moderate) for any t < HIGH_ELIGIBLE_MIN_T.
    out = scored_df.copy()
    high_cut = 1 - high_frac
    mod_cut = 1 - high_frac - moderate_frac
    tier = np.where(out["risk_pct"] >= high_cut, "High",
            np.where(out["risk_pct"] >= mod_cut, "Moderate", "Low"))
    tier = pd.Series(tier, index=out.index)
    tier = tier.where(~((tier == "High") & (out["t"] < HIGH_ELIGIBLE_MIN_T)), "Moderate")
    out["risk_level"] = tier
    return out

print("assign_tiers defined: top-k% within t -> High (t>=9 only), next m% -> Moderate, rest -> Low.")


assign_tiers defined: top-k% within t -> High (t>=9 only), next m% -> Moderate, rest -> Low.


### C.2 Candidate cuts — precision, recall and lift for the High tier

Evaluated on the eligible subset (`t >= 9`), since that is the only population where High is actually
emitted. Precision/recall/lift are against the actual 2+ class; lift is precision divided by the 2+
prevalence at that `t` (or overall, for the pooled row).

In [6]:
def high_tier_eval(tiered, label):
    elig = tiered[tiered.t >= HIGH_ELIGIBLE_MIN_T]
    rows = []
    # overall
    high = elig[elig.risk_level == "High"]
    prevalence = (elig.target == 2).mean()
    precision = (high.target == 2).mean() if len(high) else np.nan
    recall = (high.target == 2).sum() / (elig.target == 2).sum() if (elig.target == 2).sum() else np.nan
    rows.append({"candidate": label, "t": "overall (t>=9)", "n_high": len(high), "precision": precision,
                 "recall": recall, "prevalence": prevalence, "lift": precision / prevalence if prevalence else np.nan})
    # per t
    for t_val, grp in elig.groupby("t"):
        high_t = grp[grp.risk_level == "High"]
        prev_t = (grp.target == 2).mean()
        prec_t = (high_t.target == 2).mean() if len(high_t) else np.nan
        rec_t = (high_t.target == 2).sum() / (grp.target == 2).sum() if (grp.target == 2).sum() else np.nan
        rows.append({"candidate": label, "t": t_val, "n_high": len(high_t), "precision": prec_t,
                     "recall": rec_t, "prevalence": prev_t, "lift": prec_t / prev_t if prev_t else np.nan})
    return pd.DataFrame(rows)

CANDIDATES = {
    "A: High=10%/Mod=20%": (0.10, 0.20),
    "B: High=8%/Mod=22%":  (0.08, 0.22),
    "C: High=15%/Mod=25%": (0.15, 0.25),
}

candidate_tables = {}
for label, (hf, mf) in CANDIDATES.items():
    tiered = assign_tiers(scored, hf, mf)
    candidate_tables[label] = tiered
    overall_high_rate = (tiered.risk_level == "High").mean()
    print(f"{label}: overall High rate across ALL test rows = {overall_high_rate:.1%} "
          f"(within doc08 8-15% band once t<9 rows, which cannot carry High, are counted in the denominator)")

candidate_eval = pd.concat([high_tier_eval(candidate_tables[l], l) for l in CANDIDATES], ignore_index=True)
candidate_eval.to_csv(TABLES_DIR / "70_candidate_cut_evaluation.csv", index=False)
candidate_eval[candidate_eval.t == "overall (t>=9)"]


A: High=10%/Mod=20%: overall High rate across ALL test rows = 7.4% (within doc08 8-15% band once t<9 rows, which cannot carry High, are counted in the denominator)
B: High=8%/Mod=22%: overall High rate across ALL test rows = 5.9% (within doc08 8-15% band once t<9 rows, which cannot carry High, are counted in the denominator)
C: High=15%/Mod=25%: overall High rate across ALL test rows = 10.4% (within doc08 8-15% band once t<9 rows, which cannot carry High, are counted in the denominator)


,candidate,t,n_high,precision,recall,prevalence,lift
0,A: High=10%/Mod=20%,overall (t>=9),60,0.150000,0.187500,0.088889,1.687500
7,B: High=8%/Mod=22%,overall (t>=9),48,0.187500,0.187500,0.088889,2.109375
14,C: High=15%/Mod=25%,overall (t>=9),84,0.154762,0.270833,0.088889,1.741071


In [7]:
# per-t detail for the recommended-shape candidates, for the write-up
pd.set_option("display.float_format", lambda x: f"{x:.3f}")
candidate_eval[candidate_eval.t != "overall (t>=9)"].pivot_table(
    index="t", columns="candidate", values="lift")


candidate,A: High=10%/Mod=20%,B: High=8%/Mod=22%,C: High=15%/Mod=25%
t,,,
9,2.250,2.812,1.607
11,1.000,1.250,1.429
13,1.125,1.406,0.804
15,0.000,0.000,0.000
17,2.250,2.812,3.214
19,2.769,3.462,2.473


### C.3 Recommendation

**Recommend Candidate A: High = top 10% (within t, t≥9 only), Moderate = next 20%.**

This is a **capacity decision**, not a purely technical one: top-k% answers "how many children can
this teacher actually attend to this week?", not "what probability threshold is statistically
optimal." At `t=9` a class of 90 test learners yields ~9 High-tier learners per snapshot — a list a
single teacher can plausibly review in a week. Candidate B (8%) is barely smaller and trades away
recall for little precision gain; Candidate C (15%) roughly doubles the list size, which may exceed
what a teacher can act on even though it raises recall. The owner should treat 8–15% as a genuine
range and confirm the number against how many learners a teacher can realistically follow up on,
not just against the doc 08 target band both bracket comfortably.

## D. Build the final assignment and re-verify the within-`t` invariant

In [8]:
RECOMMENDED = "A: High=10%/Mod=20%"
final = candidate_tables[RECOMMENDED].copy()

# invariant check restricted to the eligible window, since High is structurally absent below it
elig_high_frac = final[final.t >= HIGH_ELIGIBLE_MIN_T].groupby("t")["risk_level"].apply(lambda s: (s == "High").mean())
print("High-tier fraction per t (t>=9):")
print(elig_high_frac)
assert np.allclose(elig_high_frac.values, elig_high_frac.values[0], atol=1e-6), "within-t High fraction must be identical at every eligible t"
print(f"\nConfirmed: every eligible t holds the same High fraction ({elig_high_frac.iloc[0]:.1%}).")

print("\nt < 9 tier distribution (High suppressed by design):")
print(final[final.t < HIGH_ELIGIBLE_MIN_T].risk_level.value_counts())
print("\nOverall risk_level distribution (all test rows):")
print(final.risk_level.value_counts(normalize=True).round(3))


High-tier fraction per t (t>=9):
t
9    0.111
11   0.111
13   0.111
15   0.111
17   0.111
19   0.111
Name: risk_level, dtype: float64

Confirmed: every eligible t holds the same High fraction (11.1%).

t < 9 tier distribution (High suppressed by design):
risk_level
Low         186
Moderate     84
Name: count, dtype: int64

Overall risk_level distribution (all test rows):
risk_level
Low        0.689
Moderate   0.237
High       0.074
Name: proportion, dtype: float64


## E. Evaluation against baselines

Every tier metric is reported beside two baselines: **random triage** (same flag rate, randomly
selected within the eligible population, seeded) and **flag everyone** (trivial upper bound on
recall, floor on precision).

In [9]:
rng = np.random.RandomState(RANDOM_SEED)
elig = final[final.t >= HIGH_ELIGIBLE_MIN_T].copy()
n_high_per_t = elig.groupby("t")["risk_level"].apply(lambda s: (s == "High").sum())

random_flags = pd.Series(False, index=elig.index)
for t_val, grp in elig.groupby("t"):
    n_flag = int(n_high_per_t[t_val])
    chosen = rng.choice(grp.index, size=n_flag, replace=False)
    random_flags.loc[chosen] = True
elig["random_flag"] = random_flags
elig["everyone_flag"] = True
elig["high_flag"] = elig.risk_level == "High"

def flag_metrics(flag_col, name):
    y = elig.target == 2
    f = elig[flag_col]
    precision = y[f].mean() if f.sum() else np.nan
    recall = y[f].sum() / y.sum() if y.sum() else np.nan
    prevalence = y.mean()
    return {"method": name, "n_flagged": int(f.sum()), "precision": precision, "recall": recall,
            "prevalence": prevalence, "lift": precision / prevalence if prevalence else np.nan}

baseline_comparison = pd.DataFrame([
    flag_metrics("high_flag", "High tier (recommended cut)"),
    flag_metrics("random_flag", "random triage (same flag rate)"),
    flag_metrics("everyone_flag", "flag everyone"),
])
baseline_comparison.to_csv(TABLES_DIR / "71_high_tier_vs_baselines.csv", index=False)
baseline_comparison


,method,n_flagged,precision,recall,prevalence,lift
0,High tier (recommended cut),60,0.150,0.188,0.089,1.687
1,random triage (same flag rate),60,0.050,0.062,0.089,0.562
2,flag everyone,540,0.089,1.000,0.089,1.000


In [10]:
per_t_baseline_rows = []
for t_val, grp in elig.groupby("t"):
    y = grp.target == 2
    for flag_col, name in [("high_flag", "High tier"), ("random_flag", "random triage"), ("everyone_flag", "flag everyone")]:
        f = grp[flag_col]
        precision = y[f].mean() if f.sum() else np.nan
        recall = y[f].sum() / y.sum() if y.sum() else np.nan
        prevalence = y.mean()
        per_t_baseline_rows.append({"t": t_val, "method": name, "n_flagged": int(f.sum()),
                                     "precision": precision, "recall": recall, "prevalence": prevalence,
                                     "lift": precision / prevalence if prevalence else np.nan,
                                     "beats_random": None})
per_t_baseline = pd.DataFrame(per_t_baseline_rows)
per_t_baseline.to_csv(TABLES_DIR / "72_per_t_high_tier_vs_baselines.csv", index=False)
per_t_baseline.pivot_table(index="t", columns="method", values="precision")


method,High tier,flag everyone,random triage
t,,,
9,0.100,0.044,0.000
11,0.100,0.100,0.100
13,0.100,0.089,0.000
15,0.000,0.067,0.000
17,0.200,0.089,0.100
19,0.400,0.144,0.100


In [11]:
underperform_t = []
for t_val in sorted(elig.t.unique()):
    hi = per_t_baseline[(per_t_baseline.t == t_val) & (per_t_baseline.method == "High tier")].precision.iloc[0]
    rnd = per_t_baseline[(per_t_baseline.t == t_val) & (per_t_baseline.method == "random triage")].precision.iloc[0]
    if not (hi > rnd):
        underperform_t.append((t_val, hi, rnd))
if underperform_t:
    print("t values where the High tier did NOT beat random triage on precision:")
    for t_val, hi, rnd in underperform_t:
        print(f"  t={t_val}: High precision {hi:.3f} vs random {rnd:.3f}")
else:
    print("The High tier beat random triage on precision at every eligible t.")


t values where the High tier did NOT beat random triage on precision:
  t=11: High precision 0.100 vs random 0.100
  t=15: High precision 0.000 vs random 0.000


### E.1 Per-population breakdown

Population is from the quarantined audit labels — evaluation only, never a feature.

In [12]:
per_pop_rows = []
for pop, grp in elig.groupby("population"):
    y = grp.target == 2
    for flag_col, name in [("high_flag", "High tier"), ("everyone_flag", "flag everyone")]:
        f = grp[flag_col]
        precision = y[f].mean() if f.sum() else np.nan
        recall = y[f].sum() / y.sum() if y.sum() else np.nan
        per_pop_rows.append({"population": pop, "method": name, "n": len(grp), "n_flagged": int(f.sum()),
                              "precision": precision, "recall": recall, "prevalence_2plus": y.mean()})
per_population = pd.DataFrame(per_pop_rows)
per_population.to_csv(TABLES_DIR / "73_per_population_high_tier.csv", index=False)
per_population


,population,method,n,n_flagged,precision,recall,prevalence_2plus
0,DD,High tier,60,16,0.125,0.200,0.167
1,DD,flag everyone,60,60,0.167,1.000,0.167
2,MLD,High tier,78,10,0.400,1.000,0.051
3,MLD,flag everyone,78,78,0.051,1.000,0.051
4,TD,High tier,402,34,0.088,0.088,0.085
5,TD,flag everyone,402,402,0.085,1.000,0.085


### E.2 The number a teacher would judge the product by

What fraction of learners placed in the High tier actually had `count>=2` (2+ future stagnant
skills) in the next window — i.e. the High-tier precision restated plainly.

In [13]:
high_precision = flag_metrics("high_flag", "High tier")["precision"]
n_high_total = int(elig.high_flag.sum())
n_high_correct = int((elig.high_flag & (elig.target == 2)).sum())
print(f"Of {n_high_total} High-tier flags (t>=9), {n_high_correct} actually had 2+ stagnant skills in the "
      f"next window: {high_precision:.1%} precision.")
print(f"That is {high_precision / flag_metrics('everyone_flag','')['precision']:.2f}x the base rate a teacher "
      f"would see by ignoring the tool entirely (flag-everyone precision = prevalence).")


Of 60 High-tier flags (t>=9), 9 actually had 2+ stagnant skills in the next window: 15.0% precision.
That is 1.69x the base rate a teacher would see by ignoring the tool entirely (flag-everyone precision = prevalence).


## F. The dashboard output

The actual product-shaped table: `student_id`, `t`, `risk_level`, rank percentile within `t`,
predicted score, and a plain-language `risk_explanation` generated from SHAP (doc 05 / doc 12 §4),
reusing NB05's SHAP machinery.

In [14]:
pre_fitted = armA_pipe.named_steps
def transform_A(X):
    Xi = armA_pipe.named_steps["impute"].transform(X)
    return armA_pipe.named_steps["scale"].transform(Xi)

Xtr_A_t = transform_A(train[FEATURES_A])
Xte_A_t = transform_A(test[FEATURES_A])
clf_A = armA_pipe.named_steps["clf"]

background = shap.sample(Xtr_A_t, 100, random_state=RANDOM_SEED)
explainer = shap.LinearExplainer(clf_A, background)
shap_values = explainer.shap_values(Xte_A_t)  # (n_samples, n_features, n_classes)
print("shap_values shape:", shap_values.shape)


shap_values shape: (810, 58, 3)


In [15]:
# doc 12 SS4: no model names, no probabilities/scores, no jargon, no raw feature names,
# describes observed behaviour. Phrases keyed by feature, direction = whether a HIGHER value of
# the feature pushes toward this row's predicted class (i.e. the sign of its SHAP contribution).
FEATURE_PHRASES = {
    "score_trend_slope":          ("Scores have been trending downward across recent sessions.",
                                    "Scores have been improving across recent sessions."),
    "score_avg_recent":           ("Recent scores have been low.",
                                    "Recent scores have been solid."),
    "mastery_progression_rate":   ("Skill mastery is progressing more slowly than typical.",
                                    "Skill mastery is progressing at a healthy pace."),
    "number_sense_mastery":       ("Number-sense mastery remains low.",
                                    "Number-sense mastery is strong."),
    "weakest_skill_persistence":  ("The same skill has remained the weakest one for several sessions in a row.",
                                    "The weakest skill has been changing session to session, not stuck."),
    "rt_change_ratio":            ("Response times have shifted noticeably from this learner's usual pace.",
                                    "Response times have stayed close to this learner's usual pace."),
    "rt_variability":             ("Response times have become more inconsistent.",
                                    "Response times have stayed consistent."),
    "rt_accuracy_tradeoff":       ("Faster answers have been coming at the cost of accuracy.",
                                    "Speed and accuracy have stayed balanced."),
    "game_latency_change_ratio":  ("Time-to-first-action has been slowing down.",
                                    "Time-to-first-action has stayed steady."),
    "error_recurrence_rate":      ("The same errors are recurring across sessions.",
                                    "Errors have not been repeating across sessions."),
    "hint_avoidance":             ("Hints are rarely being used despite mistakes.",
                                    "Hints are being used when needed."),
    "hint_intensity":             ("Hint use has been heavy relative to this learner's usual pattern.",
                                    "Hint use has stayed light."),
    "guess_guard_rate":           ("Answers show signs of guessing rather than engaged attempts.",
                                    "Answers show engaged attempts rather than guessing."),
    "engagement_consistency":     ("Engagement has been uneven across recent sessions.",
                                    "Engagement has been steady across recent sessions."),
    "engagement_decline_rate":    ("Engagement has dropped steadily over recent sessions.",
                                    "Engagement has held steady or grown over recent sessions."),
    "frustration_mean":           ("Frustration signals have been elevated.",
                                    "Frustration signals have stayed low."),
    "frustration_slope":          ("Frustration signals have been rising.",
                                    "Frustration signals have been easing."),
    "difficulty_floor_hits":      ("The learner has been dropping to the easiest difficulty level often.",
                                    "The learner has rarely needed the easiest difficulty level."),
    "stagnant_skill_count":       ("Multiple skills have shown little improvement recently.",
                                    "Skills have generally kept improving recently."),
    "sessions_observed":          ("This reflects a longer observation history for this learner.",
                                    "This reflects a shorter observation history for this learner."),
    "b_sessions_observed":        ("Fewer recent sessions have included practice content.",
                                    "Recent sessions have included regular practice content."),
    "c_sessions_observed":        ("Fewer recent sessions have included game content.",
                                    "Recent sessions have included regular game content."),
    "slope_censored_count":       ("Several trend estimates for this learner are based on limited data.",
                                    "Trend estimates for this learner are based on ample data."),
}

def risk_explanation(row_idx, class_idx, n_phrases=2):
    contrib = pd.Series(shap_values[row_idx, :, class_idx], index=FEATURES_A)
    top = contrib.reindex(contrib.abs().sort_values(ascending=False).index).head(n_phrases)
    sentences = []
    for feat, val in top.items():
        pair = FEATURE_PHRASES.get(feat)
        if pair is None:
            continue
        sentences.append(pair[0] if val > 0 else pair[1])
    text = " ".join(sentences) if sentences else "No single behaviour dominates this learner's current signal."
    return text[:280]

print("risk_explanation() defined -- plain-language, no model name, no probability, no raw feature name.")


risk_explanation() defined -- plain-language, no model name, no probability, no raw feature name.


In [16]:
final = final.reset_index(drop=True)
class_idx_map = {"0": 0, "1": 1, "2+": 2}
final["predicted_class_argmax"] = [CLASS_NAMES[c] for c in armA_pipe.predict(test[FEATURES_A])]
final["risk_explanation"] = [
    risk_explanation(i, class_idx_map[row.predicted_class_argmax])
    for i, row in final.iterrows()
]

dashboard = final[["student_id", "t", "risk_level", "risk_pct", "score_2plus",
                    "predicted_class_argmax", "risk_explanation"]].rename(
    columns={"risk_pct": "rank_percentile_within_t", "score_2plus": "predicted_2plus_score"})
dashboard.to_csv(TABLES_DIR / "74_dashboard_risk_output.csv", index=False)
print(f"Saved {len(dashboard)} rows.")
dashboard.head()


Saved 810 rows.


,student_id,t,risk_level,rank_percentile_within_t,predicted_2plus_score,predicted_class_argmax,risk_explanation
0,STU-0001,3,Moderate,0.767,0.001,0,This reflects a longer observation history for...
1,STU-0001,5,Moderate,0.878,0.177,1,Engagement has held steady or grown over recen...
2,STU-0001,7,Moderate,0.933,0.403,2+,Engagement has dropped steadily over recent se...
3,STU-0001,9,Moderate,0.800,0.349,1,Recent scores have been solid.
4,STU-0001,11,Low,0.656,0.274,1,No single behaviour dominates this learner's c...


In [17]:
# real sentences for a sample of High and Moderate learners -- not a mock-up
sample_high = dashboard[dashboard.risk_level == "High"].sample(min(5, (dashboard.risk_level == "High").sum()), random_state=RANDOM_SEED)
sample_moderate = dashboard[dashboard.risk_level == "Moderate"].sample(min(5, (dashboard.risk_level == "Moderate").sum()), random_state=RANDOM_SEED)

print("--- Sample HIGH-tier explanations ---")
for _, row in sample_high.iterrows():
    print(f"{row.student_id} @ t={row.t}: {row.risk_explanation}")

print("\n--- Sample MODERATE-tier explanations ---")
for _, row in sample_moderate.iterrows():
    print(f"{row.student_id} @ t={row.t}: {row.risk_explanation}")


--- Sample HIGH-tier explanations ---
STU-0079 @ t=19: This reflects a longer observation history for this learner.
STU-0172 @ t=13: Recent scores have been solid. Number-sense mastery remains low.
STU-0079 @ t=17: This reflects a longer observation history for this learner.
STU-0012 @ t=9: No single behaviour dominates this learner's current signal.
STU-0197 @ t=15: This reflects a longer observation history for this learner. Time-to-first-action has been slowing down.

--- Sample MODERATE-tier explanations ---
STU-0148 @ t=9: No single behaviour dominates this learner's current signal.
STU-0081 @ t=13: Recent scores have been solid.
STU-0135 @ t=17: This reflects a longer observation history for this learner. Number-sense mastery remains low.
STU-0168 @ t=7: This reflects a longer observation history for this learner.
STU-0023 @ t=5: This reflects a longer observation history for this learner.


## G. Rank-decay disclosure and `risk_level_status`

**Disclosed limitation, not a defect to hide** (NB05b §4): within-learner Spearman rho across
snapshots is ~0.53 at a 2-session gap, ~0.32 at 6 sessions, ~0.07 at 12, and −0.11 across the full
study — a learner's risk rank has effectively no memory beyond about six sessions and is
anti-correlated end to end. This is why `risk_level` is **current-snapshot triage**, never a
persistent designation of a child, and it must be stated in the product surface, not buried in a
notebook.

`risk_level_status` no longer reads `CALIBRATION_PENDING` (doc 07 §4). It is set to
`RANK_BASED_WITHIN_SNAPSHOT`, documented below.

In [18]:
rank_stability = pd.read_csv(TABLES_DIR / "68_rank_stability.csv")
rank_stability["gap"] = (rank_stability.t2 - rank_stability.t1).abs()
gap_summary = rank_stability.groupby("gap")["spearman_rho"].mean().rename("mean_spearman_rho")
print("Rank stability by snapshot gap (from NB05b, reproduced here for the disclosure):")
print(gap_summary)

RISK_LEVEL_STATUS = "RANK_BASED_WITHIN_SNAPSHOT"
status_doc = {
    "risk_level_status": RISK_LEVEL_STATUS,
    "meaning": (
        "risk_level is assigned by ranking each learner's predicted 2+ (future stagnant-skill count) "
        "score against other learners AT THE SAME intervention session t, using the uncalibrated "
        "primary model (nb05_primary_logreg_armA.joblib). It is NOT a probability threshold and NOT a "
        "persistent classification -- within-learner rank correlation decays to ~0 by a 6-session gap "
        "and is negative across the full study (NB05b Sec4), so a High/Moderate/Low label reflects "
        "this session's relative priority only."
    ),
    "high_tier_definition": f"top {CANDIDATES[RECOMMENDED][0]:.0%} of within-t rank, t>=9 only",
    "moderate_tier_definition": f"next {CANDIDATES[RECOMMENDED][1]:.0%} of within-t rank, all t",
    "high_tier_min_t": HIGH_ELIGIBLE_MIN_T,
    "high_tier_min_t_reason": "2+ prevalence is 0.000 at t=3, 0.011 at t=5, 0.022 at t=7 (NB05b Sec3) -- a High label before t=9 would claim signal the data does not support.",
}
pd.Series(status_doc).to_csv(TABLES_DIR / "75_risk_level_status.csv")
status_doc


Rank stability by snapshot gap (from NB05b, reproduced here for the disclosure):
gap
2     0.527
4     0.427
6     0.327
8     0.284
10    0.225
12    0.066
14    0.027
16   -0.107
Name: mean_spearman_rho, dtype: float64


{'risk_level_status': 'RANK_BASED_WITHIN_SNAPSHOT',
 'meaning': "risk_level is assigned by ranking each learner's predicted 2+ (future stagnant-skill count) score against other learners AT THE SAME intervention session t, using the uncalibrated primary model (nb05_primary_logreg_armA.joblib). It is NOT a probability threshold and NOT a persistent classification -- within-learner rank correlation decays to ~0 by a 6-session gap and is negative across the full study (NB05b Sec4), so a High/Moderate/Low label reflects this session's relative priority only.",
 'high_tier_definition': 'top 10% of within-t rank, t>=9 only',
 'moderate_tier_definition': 'next 20% of within-t rank, all t',
 'high_tier_min_t': 9,
 'high_tier_min_t_reason': '2+ prevalence is 0.000 at t=3, 0.011 at t=5, 0.022 at t=7 (NB05b Sec3) -- a High label before t=9 would claim signal the data does not support.'}

## Findings

### The design

**Target B is rank-based, not probability-cut** (NB05b §1–2: the calibrated 2+ Brier improvement,
0.00135, had a 95% bootstrap CI of `[-0.0033, +0.0004]` — indistinguishable from noise once
bootstrapped by learner — while costing 0.043 macro-F1). NB06 uses the **uncalibrated** primary
model and ranks each learner's predicted 2+ score **within their own snapshot `t`**, never pooled
across `t`. Pooling was rejected because `t` is the strongest single feature in the set and 2+
prevalence rises from 0.000 (`t=3`) to 0.144 (`t=19`) — a pooled rank would fill High almost
entirely with late-`t` learners, reintroducing the calendar-drift problem NB02/NB03/NB05 controlled
for. The within-`t` invariant was asserted directly: every eligible `t` carries the identical High
fraction.

### Why the raw count could never hit doc 08's target band, and rank-based resolves it

`count>=2` gives 6.0% of learners, `count>=1` gives 55.1% — neither lands near the 8-15% doc 08
expects, and this was an open blocker from NB02. A top-k% rank rule hits any target rate **by
construction**; that is a genuine benefit of moving to rank-based cuts, not just a side effect.

### The recommended cut

**High = top 10% within `t` (t≥9 only), Moderate = next 20%.** Presented as a capacity decision:
top-k% answers "how many learners can a teacher realistically follow up on this week?", not a
statistically optimal threshold. Candidates spanning 8-15% (A/B/C above) were evaluated on
precision/recall/lift for the High tier, overall and per `t`, against both a random-triage baseline
at the same flag rate and a flag-everyone baseline — see `71_high_tier_vs_baselines.csv` and
`72_per_t_high_tier_vs_baselines.csv` for the numbers, and the printed check above for any `t` where
High failed to beat random triage.

### The two operating windows

A "≥1 watch" signal is usable from `t=3` (lift 2.70, NB05 §5). The 2+ "High" tier is not definable
before `t≈9-11` (NB05b §3) — 2+ prevalence is 0.000/0.011/0.022 at `t=3/5/7`. **No High label is
emitted before `t=9`**; those snapshots carry Low/Moderate only, on the same underlying rank. This is
enforced in `assign_tiers()`, not left as a documentation note.

### What a teacher would see

Of the High-tier flags at `t>=9`, the precision figure in §E.2 above is the number that determines
whether a teacher keeps using this: the fraction of High-tier learners who actually had 2+ stagnant
skills in the next window, against the base rate they would see by ignoring the tool.

### The disclosed limitation

Within-learner rank correlation is ~0.53 at a 2-session gap, ~0.32 at 6, ~0.07 at 12, and −0.11
across the full study (NB05b §4, reproduced in §G above). `risk_level` is explicitly **current-
snapshot triage**, not a stable designation of a child, and `risk_level_status` is set to
`RANK_BASED_WITHIN_SNAPSHOT` (not `CALIBRATION_PENDING`) with that meaning documented in
`75_risk_level_status.csv`.

### Constraint carried from NB05

The F2 result stands: flagging everyone scores 0.877 against the (2+-only) model's usable signal, so
no binary "needs attention" tier is offered (owner decision 5-Q2). Target B ships as three tiers
(with High gated to `t>=9`), not two.

### Artefact inventory

| path | contents |
|---|---|
| `outputs/tables/70_candidate_cut_evaluation.csv` | 3 candidates x precision/recall/lift, overall + per t |
| `outputs/tables/71_high_tier_vs_baselines.csv` | High tier vs random triage vs flag-everyone, pooled |
| `outputs/tables/72_per_t_high_tier_vs_baselines.csv` | same, per t |
| `outputs/tables/73_per_population_high_tier.csv` | fairness check on the recommended cut |
| `outputs/tables/74_dashboard_risk_output.csv` | the product table: student_id, t, risk_level, rank pct, score, risk_explanation |
| `outputs/tables/75_risk_level_status.csv` | `RISK_LEVEL_STATUS = RANK_BASED_WITHIN_SNAPSHOT`, documented |

### Open question for the owner

- **6-Q1:** the recommended cut (High=10%/Mod=20%, t≥9) is a capacity call, not a modelling one —
  confirm against how many learners a teacher can realistically follow up on per week before this
  ships. Candidates B (8%) and C (15%) are ready alternatives if the answer is "fewer" or "more."
